In [29]:
import pandas as pd
import numpy as np
import requests
import os
import dotenv
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import plotly.colors as colors
import textwrap
from plotly.subplots import make_subplots
from datetime import datetime
from scipy.stats import gaussian_kde



In [2]:
dotenv.load_dotenv(override=True)

True

In [3]:
token = os.getenv("REDCAP_API_KEY")

In [4]:
id_list = [
    "56-1",
    "57-1",
    "78-1",
    "78-2",
    "79-1"
]

In [5]:
data = {
    'token': token,
    'content': 'record',
    'action': 'export',
    'format': 'json',
    'type': 'eav',
    # 'records[]': id_list,  # Passa a lista diretamente aqui
    'fields[]': ['record_id'],
    'forms[]': ['demografia'],
    'events[]': ['visita_1__screenin_arm_1'],
    'rawOrLabel': 'label',
    'rawOrLabelHeaders': 'label',
    'exportCheckboxLabel': 'false',
    'exportSurveyFields': 'false',
    'exportDataAccessGroups': 'false',
    'returnFormat': 'json'
}

In [6]:
r = requests.post('https://svriglobal.org/redcap/api/', data=data)
print('HTTP Status:', r.status_code)
print(r.json())

HTTP Status: 200
[{'record': '56-1', 'redcap_event_name': 'Visita 1 - Screening', 'field_name': 'demografia_centro', 'value': 'Hospital e Maternidade Dr. Christóvão da Gama S.A'}, {'record': '56-1', 'redcap_event_name': 'Visita 1 - Screening', 'field_name': 'demografia_complete', 'value': 'Complete'}, {'record': '56-1', 'redcap_event_name': 'Visita 1 - Screening', 'field_name': 'demografia_data_nascimento', 'value': '2005-02-11'}, {'record': '56-1', 'redcap_event_name': 'Visita 1 - Screening', 'field_name': 'demografia_dt_visita', 'value': '2025-12-16'}, {'record': '56-1', 'redcap_event_name': 'Visita 1 - Screening', 'field_name': 'demografia_escolaridade', 'value': 'Ensino Médio Completo'}, {'record': '56-1', 'redcap_event_name': 'Visita 1 - Screening', 'field_name': 'demografia_etnia', 'value': 'Hispânico ou Latino'}, {'record': '56-1', 'redcap_event_name': 'Visita 1 - Screening', 'field_name': 'demografia_iniciais', 'value': 'WBL'}, {'record': '56-1', 'redcap_event_name': 'Visita 1 

In [7]:
df = pd.DataFrame(r.json())

In [8]:
# filtrar df onde field_name = demografia_centro
distribuicao_sites = df[df['field_name'] == 'demografia_centro']

In [9]:
# contar a distribuição de sites
distribuicao_sites_count = distribuicao_sites['value'].value_counts()
print(distribuicao_sites_count)

value
Centro de Oncologia do Paraná                                 21
Irmandade da Santa Casa de Misericórdia de Santos             19
Hospital e Maternidade Dr. Christóvão da Gama S.A             18
Clinica de Ortopedia Campo Largo S/S LTDA                     15
Centro de Investigação Clínica Ciências Médicas (CIC CMMG)     4
Name: count, dtype: int64


In [19]:
# Gráfico de barras em plotly
def quebrar_text_site(texto, largura_max=16):
    """
    Quebra o texto inserindo <br> para manter cada linha 
    com aproximadamente 'largura_max' caracteres.
    """
    return "<br>".join(textwrap.wrap(str(texto), width=largura_max))
sites_formatados = [quebrar_text_site(site, largura_max=18) for site in distribuicao_sites_count.index]
    
fig = px.bar(
    distribuicao_sites_count, 
    x=sites_formatados, 
    y=distribuicao_sites_count.values, 
    labels={'x':'', 'y':''}, 
    title='Distribuição de Sites'
)
# adicionar o valor no lado externo superior da barra, limpar o fundo e tirar a grade
fig.update_traces(texttemplate='%{y}', textposition='inside')
fig.update_layout(
    plot_bgcolor='white',
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=False)
)
#Retirar os valores do eixo Y (somente exibir os valores nas barras)
fig.update_yaxes(showticklabels=False)
fig.show()

In [11]:
# filtrar df onde field_name = demografia_sexo
distribuicao_sexo = df[df['field_name'] == 'demografia_sexo']

In [12]:
# contar a distribuição de sites
distribuicao_sexo_count = distribuicao_sexo['value'].value_counts()
print(distribuicao_sexo_count)

value
Masculino    62
Feminino     17
Name: count, dtype: int64


In [13]:
# contar percentual a distribuição de gênero
distribuicao_sexo_percentual = distribuicao_sexo_count / distribuicao_sexo_count.sum() * 100
print(distribuicao_sexo_percentual)

value
Masculino    78.481013
Feminino     21.518987
Name: count, dtype: float64


In [14]:
# Gráfico de pizza em plotly
fig = px.pie(
    distribuicao_sexo_count, 
    names=distribuicao_sexo_count.index, 
    values=distribuicao_sexo_count.values, 
    title='Distribuição de Gênero/Sexo'
)
# adicionar o valor dentro das fatias, limpar o fundo e tirar a grade
fig.update_traces(texttemplate='%{value}', textposition='inside')
fig.update_layout(
    plot_bgcolor='white'
)
fig.show()

In [15]:
# Gráfico de pizza em plotly com valores relativos
fig = px.pie(
    distribuicao_sexo_percentual, 
    names=distribuicao_sexo_percentual.index, 
    values=distribuicao_sexo_percentual.values, 
    title='Distribuição de Gênero/Sexo (Percentual)'
)
# adicionar o valor dentro das fatias, limpar o fundo e tirar a grade
fig.update_traces(texttemplate='%{value:.2f}%', textposition='inside')
fig.update_layout(
    plot_bgcolor='white'
)
fig.show()

<h1>Idade</h1>

In [16]:
distribuicao_por_idade = df[df['field_name'] == 'demografia_data_nascimento']

In [21]:
# Excluir linhas com value = 'NA'
distribuicao_por_idade = distribuicao_por_idade.dropna(subset=['value'])
distribuicao_por_idade = distribuicao_por_idade[distribuicao_por_idade['value'] != 'NA']

# Converter a coluna 'value' para datetime
distribuicao_por_idade['value'] = pd.to_datetime(distribuicao_por_idade['value'])
# alterar o nome da coluna para dt_nasc
distribuicao_por_idade = distribuicao_por_idade.rename(columns={'value': 'dt_nasc'})



In [22]:
data_atual = datetime.now()
distribuicao_por_idade['idade'] = distribuicao_por_idade['dt_nasc'].apply(lambda x: data_atual.year - x.year - ((data_atual.month, data_atual.day) < (x.month, x.day)))
distribuicao_por_idade = distribuicao_por_idade[[
    'record',
    'idade'
]]

In [23]:
# Cálculo da idade mínima arredondada para baixo para o múltiplo de 10 mais próximo
idade_minima = (min(distribuicao_por_idade['idade'])//10) *10
# Cálculo da idade máxima arredondada para cima para o múltiplo de 10 mais próximo
idade_max = ((max(distribuicao_por_idade['idade'])//10)+1) * 10
# Criação dos bins de 10 em 10 anos
bins = list(range(idade_minima, idade_max + 10, 10))
# Criação dos rótulos para cada bin de idade
labels = [f'{bins[i]}-{bins[i+1]-1}' for i in range(len(bins)-1)]
print(labels)

['10-19', '20-29', '30-39', '40-49', '50-59']


In [24]:
distribuicao_por_faixa_etaria = distribuicao_por_idade.copy()
distribuicao_por_faixa_etaria['faixa_etaria'] = pd.cut(distribuicao_por_faixa_etaria['idade'],bins=bins, labels=labels,right=False)
distribuicao_por_faixa_etaria=distribuicao_por_faixa_etaria.value_counts().reset_index()
distribuicao_por_faixa_etaria.columns = ['index', 'idade', 'faixa_etaria', 'count']

In [25]:

distribuicao_por_faixa_etaria=distribuicao_por_faixa_etaria[['faixa_etaria', 'count']]
#Excluir linhas com count = 0
distribuicao_por_faixa_etaria = distribuicao_por_faixa_etaria[distribuicao_por_faixa_etaria['count'] != 0]


In [46]:
fig = px.histogram(
    distribuicao_por_faixa_etaria,
    x='faixa_etaria',
    title='Distribuição de Idade dos Participantes',
    labels=labels,
    color_discrete_sequence=px.colors.qualitative.Set3,
    nbins=len(labels),
    category_orders={'faixa_etaria': labels},
    text_auto = True
)

fig.update_layout(
    yaxis_title='Frequência',
    bargap=0,
    plot_bgcolor='white',
)
fig.update_traces(
    marker_line_width=1.5,
    marker_line_color='black'
)



fig.show()

<h1>Raça</h1>

In [47]:
distribuicao_por_raca = df[df['field_name'] == 'demografia_raca']

In [48]:
distribuicao_pop_por_raca = distribuicao_por_raca.copy()
distribuicao_pop_por_raca = distribuicao_pop_por_raca['value'].value_counts()
print(distribuicao_pop_por_raca)

value
Branco             49
Morena ou Parda    21
Negra ou Preta      9
Name: count, dtype: int64


In [51]:
distribuicao_relativa_pop_por_raca = distribuicao_pop_por_raca / distribuicao_pop_por_raca.sum()*100
#imprimir no formato xxx.xx%
print(distribuicao_relativa_pop_por_raca.map(lambda percentual: f"{percentual:.2f}%"))

value
Branco             62.03%
Morena ou Parda    26.58%
Negra ou Preta     11.39%
Name: count, dtype: str


In [54]:
fig = px.bar(
    x=distribuicao_pop_por_raca.index,
    y=distribuicao_pop_por_raca,
    text=distribuicao_pop_por_raca
)
fig.show()

In [57]:
fig = px.pie(
    names = distribuicao_relativa_pop_por_raca.index,
    values = distribuicao_relativa_pop_por_raca.values,
    title='Distribuição de Raça (Percentual)'
)
# adicionar o valor dentro das fatias, limpar o fundo e tirar a grade
fig.update_traces(texttemplate='%{value:.2f}%', textposition='inside')
fig.update_layout(
    plot_bgcolor='white'
)
fig.show()